# AlpCAN CT Pipeline: Uctan Uca Nodul Analizi

**Amac:** Notebook 06 (segmentasyon) ve Notebook 07 (karakterizasyon) modellerini birlestirerek
tek bir CT analiz pipeline'i olusturmak.

**Icerikler:**
1. Kurulum ve Konfiguerasyon
2. Model Mimari Tanimlari (U-Net + ResNet-50 CBAM)
3. Model Yukleme
4. Pipeline Fonksiyonlari
5. Test Hastalari Uzerinde Pipeline Calistirma
6. Pipeline Metrikleri
7. Gorsellestirme
8. Klinik Karar Destegi
9. Cikti ve Ozet

**Modeller:**
- **Segmentasyon:** U-Net (ResNet-34 encoder, smp) -- Dice: 0.622
- **Karakterizasyon:** ResNet-50 + CBAM -- AUC: 0.977

**GPU:** Kaggle T4 16GB  
**Dataset:** `zhangweiled/lidcidri`, `ridvancebec/alpcan-ct-model-weights`

In [ ]:
# === Kurulum ===
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report, accuracy_score
)

# Gorsellestirme ayarlari
sns.set_style('darkgrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print(f"PyTorch: {torch.__version__}")
print(f"SMP: {smp.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Bellek: {mem_gb:.1f} GB")

In [ ]:
# === Konfiguerasyon ===
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Veri seti yollari
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")
WEIGHTS_DIR = Path("/kaggle/input/alpcan-ct-model-weights")

# Alternatif yol arama
if not DATA_DIR.exists():
    for candidate in INPUT_DIR.rglob("LIDC-IDRI-0001"):
        DATA_DIR = candidate.parent
        break

# Model agirliklari
SEG_WEIGHTS = WEIGHTS_DIR / "ct_seg_best_model.pth"
CHAR_WEIGHTS = WEIGHTS_DIR / "ct_char_best_model.pth"

# NB07 tahminleri (karsilastirma icin)
NB07_PREDS_PATH = WEIGHTS_DIR / "ct_char_predictions.csv"

# Pipeline parametreleri
SEG_IMG_SIZE = 256       # Segmentasyon girdi boyutu
CHAR_PATCH_SIZE = 128    # Karakterizasyon girdi boyutu
SEG_THRESHOLD = 0.5      # Segmentasyon mask threshold
MIN_NODULE_AREA = 10     # Minimum nodul alani (piksel)
SUSP_THRESHOLD = 0.5     # Suspicious prediction threshold

# Lung-RADS esleme
RISK_TO_LUNGRADS = {
    0: '2',    # Dusuk Risk -> Lung-RADS 2
    1: '3',    # Orta-Dusuk Risk -> Lung-RADS 3
    2: '4A',   # Orta-Yuksek Risk -> Lung-RADS 4A
    3: '4B',   # Yuksek Risk -> Lung-RADS 4B
}

LUNGRADS_DESCRIPTIONS = {
    '2':  'Benign -- Rutin tarama',
    '3':  'Muhtemelen Benign -- 6 aylik takip BT',
    '4A': 'Supheli -- 3 aylik takip BT veya PET/CT',
    '4B': 'Cok Supheli -- Doku orneklemesi onerilir',
}

RISK_NAMES = ['Dusuk Risk', 'Orta-Dusuk', 'Orta-Yuksek', 'Yuksek Risk']
RISK_COLORS = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Veri dizini: {DATA_DIR}")
print(f"Veri dizini mevcut: {DATA_DIR.exists()}")
print(f"Agirlik dizini: {WEIGHTS_DIR}")
print(f"Segmentasyon agirliklari: {SEG_WEIGHTS} -- {'MEVCUT' if SEG_WEIGHTS.exists() else 'BULUNAMADI'}")
print(f"Karakterizasyon agirliklari: {CHAR_WEIGHTS} -- {'MEVCUT' if CHAR_WEIGHTS.exists() else 'BULUNAMADI'}")
print(f"NB07 tahminleri: {NB07_PREDS_PATH} -- {'MEVCUT' if NB07_PREDS_PATH.exists() else 'BULUNAMADI'}")
print(f"Device: {device}")

---
## 2. Model Mimari Tanimlari

In [ ]:
# ============================================================
# Segmentasyon Modeli: U-Net (ResNet-34 encoder)
# NB06 ile birebir ayni mimari -- segmentation_models_pytorch
# ============================================================

def build_segmentation_model():
    """NB06 ile ayni U-Net modelini olustur.
    
    smp.Unet kullanir:
      - encoder: ResNet-34
      - in_channels: 3 (grayscale -> 3 kanala kopyalanir)
      - classes: 1 (binary segmentasyon)
      - activation: None (raw logits)
    """
    model = smp.Unet(
        encoder_name='resnet34',
        encoder_weights=None,  # Agirliklari kendimiz yukleyecegiz
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model


print("Segmentasyon modeli tanimlandi: smp.Unet (ResNet-34)")
print(f"  Giris: 3 kanal, {SEG_IMG_SIZE}x{SEG_IMG_SIZE}")
print(f"  Cikis: 1 kanal, {SEG_IMG_SIZE}x{SEG_IMG_SIZE} (raw logits)")

In [ ]:
# ============================================================
# Karakterizasyon Modeli: ResNet-50 + CBAM
# NB07 ile birebir ayni mimari
# ============================================================

class ChannelAttention(nn.Module):
    """CBAM Channel Attention.
    
    Kanal bazinda onem agirliklarini ogrenir.
    Global Average Pool + Global Max Pool -> shared MLP -> sigmoid.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        attention = torch.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * attention


class SpatialAttention(nn.Module):
    """CBAM Spatial Attention."""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        concat = torch.cat([avg_out, max_out], dim=1)
        attention = torch.sigmoid(self.conv(concat))
        return x * attention


class CBAM(nn.Module):
    """Convolutional Block Attention Module (Woo et al., ECCV 2018)."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x


class ResNet50CBAM(nn.Module):
    """ResNet-50 + CBAM nodul karakterizasyon modeli.
    
    NB07 ile birebir ayni mimari.
    
    Cikislar:
    - suspicious: Binary (0/1)
    - risk_class: 4 sinif (0-3)
    - features: Ara katman ciktilari (Grad-CAM icin)
    """
    def __init__(self, n_risk_classes=4):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.cbam = CBAM(2048, reduction=16)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc_suspicious = nn.Linear(2048, 1)
        self.fc_risk = nn.Linear(2048, n_risk_classes)

    def forward(self, x):
        features = self.features(x)
        features = self.cbam(features)
        pooled = self.gap(features).flatten(1)
        pooled = self.dropout(pooled)
        suspicious = self.fc_suspicious(pooled).squeeze(-1)
        risk = self.fc_risk(pooled)
        return suspicious, risk, features


print("Karakterizasyon modeli tanimlandi: ResNet-50 + CBAM")
print(f"  Giris: 3 kanal, {CHAR_PATCH_SIZE}x{CHAR_PATCH_SIZE}")
print(f"  Cikis 1: suspicious (binary logit)")
print(f"  Cikis 2: risk_class (4 sinif logits)")

In [ ]:
# On-isleme transformlari (inference icin -- augmentation yok)

seg_transform = A.Compose([
    A.Resize(SEG_IMG_SIZE, SEG_IMG_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
])

char_transform = A.Compose([
    A.Resize(CHAR_PATCH_SIZE, CHAR_PATCH_SIZE),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
    ToTensorV2(),
])

print("On-isleme transformlari tanimlandi.")
print(f"  Segmentasyon: Resize({SEG_IMG_SIZE}) -> ImageNet Normalize -> Tensor")
print(f"  Karakterizasyon: Resize({CHAR_PATCH_SIZE}) -> ImageNet Normalize -> Tensor")

---
## 3. Model Yukleme

In [ ]:
# === Segmentasyon modeli yukleme ===
seg_model = build_segmentation_model()

seg_checkpoint = torch.load(SEG_WEIGHTS, map_location=device, weights_only=False)

# Checkpoint formatini kontrol et (model_state_dict veya direkt state_dict)
if 'model_state_dict' in seg_checkpoint:
    seg_model.load_state_dict(seg_checkpoint['model_state_dict'])
    seg_epoch = seg_checkpoint.get('epoch', -1)
    seg_dice = seg_checkpoint.get('val_dice', 0.0)
    print(f"Segmentasyon modeli yuklendi (checkpoint formati)")
    print(f"  Epoch: {seg_epoch + 1}")
    print(f"  Val Dice: {seg_dice:.4f}")
else:
    seg_model.load_state_dict(seg_checkpoint)
    print(f"Segmentasyon modeli yuklendi (direkt state_dict)")

seg_model = seg_model.to(device)
seg_model.eval()

n_seg_params = sum(p.numel() for p in seg_model.parameters())
print(f"  Parametre: {n_seg_params:,} ({n_seg_params / 1e6:.1f}M)")
print(f"  Device: {device}")

In [ ]:
# === Karakterizasyon modeli yukleme ===
char_model = ResNet50CBAM(n_risk_classes=4)

char_checkpoint = torch.load(CHAR_WEIGHTS, map_location=device, weights_only=False)

if 'model_state_dict' in char_checkpoint:
    char_model.load_state_dict(char_checkpoint['model_state_dict'])
    char_epoch = char_checkpoint.get('epoch', -1)
    char_auc = char_checkpoint.get('val_auc', 0.0)
    print(f"Karakterizasyon modeli yuklendi (checkpoint formati)")
    print(f"  Epoch: {char_epoch + 1}")
    print(f"  Val AUC: {char_auc:.4f}")
else:
    char_model.load_state_dict(char_checkpoint)
    print(f"Karakterizasyon modeli yuklendi (direkt state_dict)")

char_model = char_model.to(device)
char_model.eval()

n_char_params = sum(p.numel() for p in char_model.parameters())
print(f"  Parametre: {n_char_params:,} ({n_char_params / 1e6:.1f}M)")
print(f"  Device: {device}")

# Toplam model ozeti
print(f"\n{'=' * 60}")
print(f"Model Ozeti")
print(f"{'=' * 60}")
print(f"  Segmentasyon: U-Net (ResNet-34) -- {n_seg_params/1e6:.1f}M parametre")
print(f"  Karakterizasyon: ResNet-50+CBAM -- {n_char_params/1e6:.1f}M parametre")
print(f"  Toplam: {(n_seg_params + n_char_params)/1e6:.1f}M parametre")

---
## 4. Pipeline Fonksiyonlari

In [ ]:
def segment_nodule(image_np, model, transform, threshold=SEG_THRESHOLD):
    """Tek bir goruntude nodul segmentasyonu yap.
    
    Args:
        image_np: (H, W) grayscale numpy array (uint8)
        model: Segmentasyon modeli (eval modda)
        transform: Albumentations transform
        threshold: Sigmoid ciktisi icin esik
    
    Returns:
        pred_mask: (H_orig, W_orig) binary maske (bool)
        pred_prob: (SEG_IMG_SIZE, SEG_IMG_SIZE) olasilik haritasi
    """
    h_orig, w_orig = image_np.shape[:2]
    
    # 3 kanala kopyala
    img_rgb = np.stack([image_np, image_np, image_np], axis=-1)
    
    # Transform uygula
    augmented = transform(image=img_rgb)
    img_tensor = augmented['image'].unsqueeze(0).to(device)  # (1, 3, 256, 256)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            output = model(img_tensor)  # (1, 1, 256, 256)
    
    # Sigmoid olasilik
    prob = torch.sigmoid(output).cpu().numpy()[0, 0]  # (256, 256)
    
    # Binary maske
    mask_256 = (prob > threshold)
    
    # Orijinal boyuta donustur
    if h_orig != SEG_IMG_SIZE or w_orig != SEG_IMG_SIZE:
        mask_pil = Image.fromarray(mask_256.astype(np.uint8) * 255)
        mask_pil = mask_pil.resize((w_orig, h_orig), Image.NEAREST)
        pred_mask = np.array(mask_pil) > 127
    else:
        pred_mask = mask_256
    
    return pred_mask, prob


def extract_nodule_candidates(pred_mask, min_area=MIN_NODULE_AREA):
    """Binary maskeden nodul adaylarini cikar (connected components).
    
    Args:
        pred_mask: (H, W) binary maske
        min_area: Minimum piksel alani
    
    Returns:
        nodules: list of dict
    """
    if pred_mask is None or pred_mask.max() == 0:
        return []
    
    labeled, n_components = ndimage.label(pred_mask.astype(np.int32))
    nodules = []
    
    for i in range(1, n_components + 1):
        component = (labeled == i)
        area = int(component.sum())
        
        if area < min_area:
            continue
        
        center = ndimage.center_of_mass(component)
        rows = np.any(component, axis=1)
        cols = np.any(component, axis=0)
        row_indices = np.where(rows)[0]
        col_indices = np.where(cols)[0]
        
        if len(row_indices) == 0 or len(col_indices) == 0:
            continue
        
        y1, y2 = int(row_indices[0]), int(row_indices[-1])
        x1, x2 = int(col_indices[0]), int(col_indices[-1])
        diameter = np.sqrt(area / np.pi) * 2
        
        nodules.append({
            'center': (float(center[0]), float(center[1])),
            'area': area,
            'bbox': (y1, x1, y2, x2),
            'diameter_px': float(diameter),
            'mask': component,
        })
    
    return nodules


print("Segmentasyon ve connected component fonksiyonlari tanimlandi.")

In [ ]:
def characterize_nodule(image_np, nodule_info, model, transform):
    """Tek bir nodul adayini siniflandir.
    
    Args:
        image_np: (H, W) grayscale goruntu
        nodule_info: extract_nodule_candidates'ten gelen dict
        model: Karakterizasyon modeli
        transform: Albumentations transform
    
    Returns:
        result: dict (suspicious_prob, risk_class, risk_probs, lung_rads)
    """
    cy, cx = nodule_info['center']
    cy, cx = int(cy), int(cx)
    
    # Nodul bolgesini crop (2x context ile)
    bbox = nodule_info['bbox']
    h = bbox[2] - bbox[0]
    w = bbox[3] - bbox[1]
    margin = max(h, w)
    half = max(margin, 32)  # Minimum 32 piksel context
    
    y1 = max(0, cy - half)
    y2 = min(image_np.shape[0], cy + half)
    x1 = max(0, cx - half)
    x2 = min(image_np.shape[1], cx + half)
    
    patch = image_np[y1:y2, x1:x2]
    
    # Bos patch kontrolu
    if patch.size == 0 or patch.shape[0] < 4 or patch.shape[1] < 4:
        patch = np.zeros((CHAR_PATCH_SIZE, CHAR_PATCH_SIZE), dtype=np.uint8)
    
    # 3 kanala kopyala
    patch_rgb = np.stack([patch, patch, patch], axis=-1)
    
    # Transform
    augmented = transform(image=patch_rgb)
    patch_tensor = augmented['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            susp_logit, risk_logit, _ = model(patch_tensor)
    
    # Tahminleri isle
    susp_prob = torch.sigmoid(susp_logit).cpu().item()
    risk_probs = F.softmax(risk_logit, dim=1).cpu().numpy()[0]
    risk_class = int(risk_logit.argmax(dim=1).cpu().item())
    
    # Lung-RADS esleme
    lung_rads = RISK_TO_LUNGRADS.get(risk_class, '?')
    
    return {
        'suspicious_prob': susp_prob,
        'suspicious': int(susp_prob >= SUSP_THRESHOLD),
        'risk_class': risk_class,
        'risk_name': RISK_NAMES[risk_class],
        'risk_probs': risk_probs.tolist(),
        'lung_rads': lung_rads,
        'lung_rads_desc': LUNGRADS_DESCRIPTIONS.get(lung_rads, ''),
        'patch_bbox': (y1, x1, y2, x2),
    }


print("Nodul karakterizasyon fonksiyonu tanimlandi.")

In [ ]:
def compute_dice_score(pred_mask, gt_mask):
    """Iki binary maske arasindaki Dice skoru hesapla."""
    pred = pred_mask.astype(bool).flatten()
    gt = gt_mask.astype(bool).flatten()
    intersection = (pred & gt).sum()
    if pred.sum() + gt.sum() == 0:
        return 1.0  # Ikisi de bos -> tam eslesme
    return 2.0 * intersection / (pred.sum() + gt.sum())


def get_consensus_mask(mask_dirs, img_name, target_shape):
    """Annotator maskelerinden konsensus maske olustur."""
    combined = np.zeros(target_shape, dtype=np.float32)
    n = 0
    for mdir in mask_dirs:
        mask_file = Path(mdir) / img_name
        if mask_file.exists():
            m = np.array(Image.open(mask_file).convert('L'))
            if m.shape == target_shape:
                combined += (m > 0).astype(np.float32)
                n += 1
            else:
                m_resized = np.array(
                    Image.open(mask_file).convert('L').resize(
                        (target_shape[1], target_shape[0]), Image.NEAREST
                    )
                )
                combined += (m_resized > 0).astype(np.float32)
                n += 1
    if n > 0:
        return (combined / n >= 0.5).astype(np.uint8)
    return np.zeros(target_shape, dtype=np.uint8)


def run_pipeline(patient_dir, seg_model, char_model,
                 seg_transform, char_transform,
                 max_slices=None, verbose=False):
    """Bir hasta icin tam pipeline calistir.
    
    Adimlar:
      1. Tum nodul dizinlerini tara
      2. Her nodul icin orta dilimi segment et
      3. Connected components ile nodul adaylarini cikar
      4. Her aday icin karakterizasyon yap
      5. GT maskeleri ile karsilastir
    
    Args:
        patient_dir: Hasta dizini (Path)
        seg_model, char_model: Modeller
        seg_transform, char_transform: Transformlar
        max_slices: Nodul basina maksimum dilim (None = hepsi)
        verbose: Detayli cikti
    
    Returns:
        results: list of dict
        timing: dict (her asamanin suresi)
    """
    patient_id = patient_dir.name
    results = []
    timing = {'segment': 0, 'extract': 0, 'characterize': 0, 'total': 0}
    t_start = time.time()
    
    nodule_dirs = sorted([d for d in patient_dir.iterdir() if d.is_dir()])
    
    if not nodule_dirs:
        timing['total'] = time.time() - t_start
        return results, timing
    
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs_list = sorted(
            [d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")]
        )
        
        if not img_dir.exists() or not mask_dirs_list:
            continue
        
        img_files = sorted(img_dir.glob("*.png"))
        if not img_files:
            continue
        
        # Orta dilimi sec (nodul en belirgin oldugu dilim)
        mid_idx = len(img_files) // 2
        img_file = img_files[mid_idx]
        
        # Goruntu oku
        image_np = np.array(Image.open(img_file).convert('L'))
        
        # GT konsensus maskesi
        gt_mask = get_consensus_mask(
            [str(m) for m in mask_dirs_list],
            img_file.name,
            image_np.shape
        )
        gt_area = int(gt_mask.sum())
        gt_diameter = np.sqrt(gt_area / np.pi) * 2 if gt_area > 0 else 0
        
        # === Adim 1: Segmentasyon ===
        t_seg = time.time()
        pred_mask, pred_prob = segment_nodule(image_np, seg_model, seg_transform)
        timing['segment'] += time.time() - t_seg
        
        # Dice skoru hesapla
        dice = compute_dice_score(pred_mask, gt_mask)
        
        # === Adim 2: Connected Components ===
        t_ext = time.time()
        candidates = extract_nodule_candidates(pred_mask)
        timing['extract'] += time.time() - t_ext
        
        if not candidates:
            # Nodul bulunamadi -- yine de GT bilgisi ile kaydet
            results.append({
                'patient_id': patient_id,
                'nodule': ndir.name,
                'image_path': str(img_file),
                'n_slices': len(img_files),
                'n_annotators': len(mask_dirs_list),
                'gt_area': gt_area,
                'gt_diameter_px': gt_diameter,
                'detected': False,
                'dice': dice,
                'pred_area': 0,
                'pred_diameter_px': 0.0,
                'suspicious_prob': np.nan,
                'suspicious': np.nan,
                'risk_class': np.nan,
                'risk_name': '',
                'lung_rads': '',
                'risk_probs': [],
            })
            continue
        
        # En buyuk nodul adayini sec
        best_candidate = max(candidates, key=lambda c: c['area'])
        
        # === Adim 3: Karakterizasyon ===
        t_char = time.time()
        char_result = characterize_nodule(
            image_np, best_candidate, char_model, char_transform
        )
        timing['characterize'] += time.time() - t_char
        
        results.append({
            'patient_id': patient_id,
            'nodule': ndir.name,
            'image_path': str(img_file),
            'n_slices': len(img_files),
            'n_annotators': len(mask_dirs_list),
            'gt_area': gt_area,
            'gt_diameter_px': gt_diameter,
            'detected': True,
            'dice': dice,
            'pred_area': best_candidate['area'],
            'pred_diameter_px': best_candidate['diameter_px'],
            'suspicious_prob': char_result['suspicious_prob'],
            'suspicious': char_result['suspicious'],
            'risk_class': char_result['risk_class'],
            'risk_name': char_result['risk_name'],
            'lung_rads': char_result['lung_rads'],
            'risk_probs': char_result['risk_probs'],
        })
        
        if verbose:
            susp_str = 'SUPHELI' if char_result['suspicious'] else 'Benign'
            print(f"    {ndir.name}: Dice={dice:.3f}, "
                  f"{susp_str} (p={char_result['suspicious_prob']:.2f}), "
                  f"Lung-RADS {char_result['lung_rads']}")
    
    timing['total'] = time.time() - t_start
    return results, timing


print("Pipeline fonksiyonu tanimlandi.")
print("  run_pipeline(patient_dir, seg_model, char_model, ...)")

---
## 5. Test Hastalari Uzerinde Pipeline Calistirma

In [ ]:
# Tum hasta dizinlerini bul
all_patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta dizini: {len(all_patient_dirs)}")

# 20 cesitli hasta sec (farkli indekslerden)
np.random.seed(42)
n_test_patients = min(20, len(all_patient_dirs))

# Esit aralikli secim (farkli boyut/konum cesitliligi icin)
indices = np.linspace(0, len(all_patient_dirs) - 1, n_test_patients, dtype=int)
test_patient_dirs = [all_patient_dirs[i] for i in indices]

print(f"Secilen test hasta sayisi: {n_test_patients}")
print(f"Test hastalari:")
for i, pdir in enumerate(test_patient_dirs):
    nodule_count = sum(1 for d in pdir.iterdir() if d.is_dir())
    print(f"  {i+1:>2d}. {pdir.name} ({nodule_count} nodul dizini)")

In [ ]:
# Tum test hastalari uzerinde pipeline calistir
all_results = []
all_timings = []

print("=" * 70)
print("Pipeline Calistiriliyor")
print("=" * 70)

total_start = time.time()

for i, pdir in enumerate(test_patient_dirs):
    print(f"\n[{i+1}/{n_test_patients}] {pdir.name} isleniyor...", end=" ")
    
    results, timing = run_pipeline(
        pdir, seg_model, char_model,
        seg_transform, char_transform,
        verbose=True
    )
    
    all_results.extend(results)
    all_timings.append({
        'patient_id': pdir.name,
        'n_nodules': len(results),
        **timing,
    })
    
    n_detected = sum(1 for r in results if r['detected'])
    print(f"  -> {len(results)} nodul, {n_detected} tespit edildi, "
          f"{timing['total']:.2f}s")
    
    # GPU bellek temizligi
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_time = time.time() - total_start

print(f"\n{'=' * 70}")
print(f"Pipeline tamamlandi!")
print(f"  Toplam sure: {total_time:.1f}s ({total_time/60:.1f} dk)")
print(f"  Toplam nodul: {len(all_results)}")
print(f"  Tespit edilen: {sum(1 for r in all_results if r['detected'])}")
print(f"  Hasta basina ortalama sure: {total_time/n_test_patients:.2f}s")

In [ ]:
# Sonuclari DataFrame'e cevir
results_df = pd.DataFrame(all_results)

# risk_probs kolonunu ayri kolonlara ac
for i, name in enumerate(['risk_prob_low', 'risk_prob_mid_low', 'risk_prob_mid_high', 'risk_prob_high']):
    results_df[name] = results_df['risk_probs'].apply(
        lambda x: x[i] if isinstance(x, list) and len(x) > i else np.nan
    )

print(f"Sonuc DataFrame:")
print(f"  Shape: {results_df.shape}")
print(f"  Kolonlar: {list(results_df.columns)}")
print(f"\nTespit orani: {results_df['detected'].mean():.1%}")
print(f"\nHasta bazinda ozet:")
patient_summary = results_df.groupby('patient_id').agg({
    'nodule': 'count',
    'detected': 'sum',
    'dice': 'mean',
    'suspicious_prob': 'mean',
}).round(3)
patient_summary.columns = ['Nodul', 'Tespit', 'Ort.Dice', 'Ort.Susp']
print(patient_summary.to_string())

In [ ]:
# NB07 tahminleri ile karsilastirma (eger mevcutsa)
if NB07_PREDS_PATH.exists():
    nb07_df = pd.read_csv(NB07_PREDS_PATH)
    print(f"NB07 tahminleri yuklendi: {len(nb07_df)} satir")
    print(f"NB07 kolonlari: {list(nb07_df.columns)}")
    
    # Ortak noduller uzerinden karsilastirma
    # patient_id + nodule ile eslestir
    pipeline_detected = results_df[results_df['detected'] == True].copy()
    
    if 'patient_id' in nb07_df.columns and 'nodule' in nb07_df.columns:
        merged = pipeline_detected.merge(
            nb07_df[['patient_id', 'nodule', 'susp_pred', 'risk_pred', 'lung_rads_pred']],
            on=['patient_id', 'nodule'],
            how='inner',
            suffixes=('_pipeline', '_nb07')
        )
        
        if len(merged) > 0:
            print(f"\nOrtak nodul sayisi (pipeline & NB07): {len(merged)}")
            print(f"\nPipeline vs NB07 suspicious karsilastirmasi:")
            print(f"  Pipeline susp. orani: {merged['suspicious_prob'].mean():.3f}")
            print(f"  NB07 susp. orani:     {merged['susp_pred'].mean():.3f}")
            
            # Korelasyon
            valid_mask = merged['suspicious_prob'].notna() & merged['susp_pred'].notna()
            if valid_mask.sum() > 2:
                corr = np.corrcoef(
                    merged.loc[valid_mask, 'suspicious_prob'],
                    merged.loc[valid_mask, 'susp_pred']
                )[0, 1]
                print(f"  Korelasyon: {corr:.4f}")
        else:
            print("\nOrtak nodul bulunamadi.")
    else:
        print("\nNB07 DataFrame'inde patient_id/nodule kolonu bulunamadi.")
else:
    print("NB07 tahminleri bulunamadi -- karsilastirma atlanacak.")
    print(f"  Beklenen yol: {NB07_PREDS_PATH}")
    print(f"  Not: ct_char_predictions.csv dosyasini alpcan-ct-model-weights dataset'ine yukleyin.")

---
## 6. Pipeline Metrikleri

In [ ]:
# === Tespit Duyarliligi ===
n_total = len(results_df)
n_detected = int(results_df['detected'].sum())
n_missed = n_total - n_detected
detection_rate = n_detected / max(n_total, 1)

print("=" * 60)
print("Pipeline Metrikleri")
print("=" * 60)

print(f"\n--- Tespit Duyarliligi ---")
print(f"  Toplam nodul: {n_total}")
print(f"  Tespit edilen: {n_detected} ({detection_rate:.1%})")
print(f"  Kacirilan: {n_missed} ({1 - detection_rate:.1%})")

# === Segmentasyon Kalitesi ===
detected_df = results_df[results_df['detected'] == True]
if len(detected_df) > 0:
    mean_dice = detected_df['dice'].mean()
    std_dice = detected_df['dice'].std()
    median_dice = detected_df['dice'].median()
    print(f"\n--- Segmentasyon Kalitesi (tespit edilen noduller) ---")
    print(f"  Ortalama Dice: {mean_dice:.4f} +/- {std_dice:.4f}")
    print(f"  Medyan Dice: {median_dice:.4f}")
    print(f"  Min Dice: {detected_df['dice'].min():.4f}")
    print(f"  Max Dice: {detected_df['dice'].max():.4f}")
else:
    mean_dice = 0.0
    print(f"\n--- Segmentasyon Kalitesi ---")
    print(f"  Hicbir nodul tespit edilemedi.")

# === Tum dilimler icin Dice (tespit edilmeyenler dahil) ===
all_dice = results_df['dice'].mean()
print(f"\n--- Genel Dice (tum noduller) ---")
print(f"  Ortalama: {all_dice:.4f}")

In [ ]:
# === Karakterizasyon Metrikleri ===
print("\n--- Karakterizasyon Sonuclari ---")

if len(detected_df) > 0:
    # Suspicious dagilimi
    susp_counts = detected_df['suspicious'].value_counts()
    print(f"\nSuspicious dagilimi:")
    for val, count in susp_counts.items():
        label = 'Supheli' if val == 1 else 'Benign'
        pct = count / len(detected_df) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")
    
    print(f"\nOrtalama suspicious olasiligi: {detected_df['suspicious_prob'].mean():.3f}")
    
    # Risk sinifi dagilimi
    print(f"\nRisk sinifi dagilimi:")
    for rc in range(4):
        count = int((detected_df['risk_class'] == rc).sum())
        pct = count / len(detected_df) * 100
        lr = RISK_TO_LUNGRADS.get(rc, '?')
        print(f"  {RISK_NAMES[rc]:>15s} (Lung-RADS {lr}): {count:>4d} ({pct:>5.1f}%)")
    
    # Lung-RADS dagilimi
    print(f"\nLung-RADS dagilimi:")
    lr_counts = detected_df['lung_rads'].value_counts().sort_index()
    for lr, count in lr_counts.items():
        if lr in LUNGRADS_DESCRIPTIONS:
            print(f"  Lung-RADS {lr}: {count:>4d} -- {LUNGRADS_DESCRIPTIONS[lr]}")
else:
    print("  Tespit edilen nodul yok -- karakterizasyon metrikleri hesaplanamadi.")

In [ ]:
# === Confusion Matrix (Risk Sinifi) ===
# GT etiketi: nodul capina dayali (NB07 ile ayni mantik)
# Pipeline tahmininden bagimsiz GT olustur

if len(detected_df) > 0 and detected_df['risk_class'].notna().sum() > 0:
    # GT risk sinifini nodul boyutundan turet
    median_diameter = results_df['gt_diameter_px'].median()
    if median_diameter == 0:
        median_diameter = results_df.loc[
            results_df['gt_diameter_px'] > 0, 'gt_diameter_px'
        ].median()
    if pd.isna(median_diameter) or median_diameter == 0:
        median_diameter = 20.0  # Fallback
    
    def gt_risk_from_diameter(d):
        if d <= median_diameter * 0.5:
            return 0
        elif d <= median_diameter:
            return 1
        elif d <= median_diameter * 2:
            return 2
        else:
            return 3
    
    detected_df = detected_df.copy()
    detected_df['gt_risk'] = detected_df['gt_diameter_px'].apply(gt_risk_from_diameter)
    detected_df['gt_suspicious'] = (
        (detected_df['gt_diameter_px'] > median_diameter) &
        (detected_df['n_annotators'] >= 3)
    ).astype(int)
    
    # Risk confusion matrix
    valid_risk = detected_df['risk_class'].notna()
    if valid_risk.sum() > 0:
        gt_risk = detected_df.loc[valid_risk, 'gt_risk'].astype(int).values
        pred_risk = detected_df.loc[valid_risk, 'risk_class'].astype(int).values
        
        present_classes = sorted(set(gt_risk) | set(pred_risk))
        cm = confusion_matrix(gt_risk, pred_risk, labels=present_classes)
        
        risk_acc = accuracy_score(gt_risk, pred_risk)
        print(f"\n--- Risk Sinifi Performansi ---")
        print(f"  Accuracy: {risk_acc:.4f}")
        print(f"  Medyan cap: {median_diameter:.1f} piksel")
        
        # Suspicious AUC
        gt_susp = detected_df.loc[valid_risk, 'gt_suspicious'].values
        pred_susp = detected_df.loc[valid_risk, 'suspicious_prob'].values
        if len(np.unique(gt_susp)) > 1:
            auc = roc_auc_score(gt_susp, pred_susp)
            print(f"  Suspicious AUC: {auc:.4f}")
        else:
            auc = np.nan
            print(f"  Suspicious AUC: Hesaplanamadi (tek sinif)")
        
        # Confusion matrix gorsellestirme
        fig, ax = plt.subplots(figsize=(8, 6))
        present_names = [RISK_NAMES[i] for i in present_classes]
        
        im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
        ax.set_title('Risk Sinifi Confusion Matrix\n(Pipeline Tahmin vs GT)', fontweight='bold')
        ax.set_xlabel('Tahmin')
        ax.set_ylabel('Gercek')
        ax.set_xticks(range(len(present_names)))
        ax.set_yticks(range(len(present_names)))
        ax.set_xticklabels(present_names, rotation=45, ha='right')
        ax.set_yticklabels(present_names)
        
        for ii in range(cm.shape[0]):
            for jj in range(cm.shape[1]):
                ax.text(jj, ii, str(cm[ii, jj]), ha='center', va='center',
                        color='white' if cm[ii, jj] > cm.max() / 2 else 'black',
                        fontsize=14, fontweight='bold')
        
        plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'ct_pipeline_confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Kaydedildi: ct_pipeline_confusion_matrix.png")
    else:
        risk_acc = np.nan
        auc = np.nan
        print("  Gecerli risk tahmini yok.")
else:
    risk_acc = np.nan
    auc = np.nan
    median_diameter = 0
    print("  Tespit edilen nodul yok -- confusion matrix olusturulamadi.")

---
## 7. Gorsellestirme

In [ ]:
# 8 secili hasta icin gorsellestirme:
# Orijinal goruntu + segmentasyon overlay + karakterizasyon sonucu

# Cesitli sonuclara sahip hasta/nodul sec
vis_df = results_df[results_df['detected'] == True].copy()
if len(vis_df) == 0:
    vis_df = results_df.copy()

n_vis = min(8, len(vis_df))
vis_indices = np.linspace(0, len(vis_df) - 1, n_vis, dtype=int)
vis_samples = vis_df.iloc[vis_indices]

fig, axes = plt.subplots(3, n_vis, figsize=(3.5 * n_vis, 10))
if n_vis == 1:
    axes = axes.reshape(3, 1)

fig.suptitle('AlpCAN CT Pipeline -- Uctan Uca Nodul Analizi', fontsize=15, fontweight='bold', y=1.02)

for col, (_, row) in enumerate(vis_samples.iterrows()):
    # Goruntu oku
    try:
        img = np.array(Image.open(row['image_path']).convert('L'))
    except Exception:
        img = np.zeros((SEG_IMG_SIZE, SEG_IMG_SIZE), dtype=np.uint8)
    
    # Segmentasyon calistir
    pred_mask, pred_prob = segment_nodule(img, seg_model, seg_transform)
    
    # Satir 0: Orijinal goruntu
    axes[0, col].imshow(img, cmap='gray')
    patient_short = row['patient_id'].replace('LIDC-IDRI-', '')
    axes[0, col].set_title(f'{patient_short}\n{row["nodule"]}', fontsize=8)
    axes[0, col].axis('off')
    
    # Satir 1: Segmentasyon overlay
    axes[1, col].imshow(img, cmap='gray')
    if pred_mask.any():
        # Overlay rengi: supheli ise kirmizi, degil ise yesil
        is_susp = row.get('suspicious', 0)
        contour_color = 'red' if is_susp == 1 else 'lime'
        try:
            axes[1, col].contour(
                pred_mask.astype(np.float32), colors=[contour_color],
                linewidths=1.5, levels=[0.5]
            )
        except Exception:
            pass
    
    dice_val = row.get('dice', 0)
    axes[1, col].set_title(f'Dice: {dice_val:.3f}', fontsize=8)
    axes[1, col].axis('off')
    
    # Satir 2: Karakterizasyon sonucu
    axes[2, col].imshow(img, cmap='gray')
    if pred_mask.any():
        # Hafif maske overlay
        overlay = np.zeros((*img.shape, 4), dtype=np.float32)
        if is_susp == 1:
            overlay[pred_mask, 0] = 1.0  # Kirmizi
            overlay[pred_mask, 3] = 0.3
        else:
            overlay[pred_mask, 1] = 1.0  # Yesil
            overlay[pred_mask, 3] = 0.3
        axes[2, col].imshow(overlay)
    
    if row['detected']:
        lr = row.get('lung_rads', '?')
        sp = row.get('suspicious_prob', 0)
        rn = row.get('risk_name', '')
        axes[2, col].set_title(f'LR-{lr} | S:{sp:.2f}\n{rn}', fontsize=7, fontweight='bold')
    else:
        axes[2, col].set_title('Tespit edilemedi', fontsize=7, color='gray')
    axes[2, col].axis('off')

# Satir etiketleri
axes[0, 0].set_ylabel('Orijinal', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Segmentasyon', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Karakterizasyon', fontsize=11, fontweight='bold')

# Legend
legend_patches = [
    mpatches.Patch(color='lime', alpha=0.7, label='Benign'),
    mpatches.Patch(color='red', alpha=0.7, label='Supheli'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_pipeline_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Kaydedildi: ct_pipeline_visualization.png")

In [ ]:
# Pipeline akis diyagrami (metin bazli)
print("=" * 70)
print("AlpCAN CT Nodul Analizi Pipeline Akisi")
print("=" * 70)
print()
print("  BT Goruntusu (DICOM / PNG)")
print("        |")
print("        v")
print("  +------------------------------------------+")
print("  |  1. ON-ISLEME                            |")
print(f"  |     Resize: {SEG_IMG_SIZE}x{SEG_IMG_SIZE}, Normalize (ImageNet) |")
print("  |     Grayscale -> 3 kanal                 |")
print("  +------------------------------------------+")
print("        |")
print("        v")
print("  +------------------------------------------+")
print("  |  2. SEGMENTASYON (U-Net + ResNet-34)     |")
print("  |     Giris: 3ch x 256 x 256               |")
print("  |     Cikis: 1ch x 256 x 256 (maske)       |")
print(f"  |     Threshold: {SEG_THRESHOLD}                        |")
print("  +------------------------------------------+")
print("        |")
print("        v")
print("  +------------------------------------------+")
print("  |  3. NODUL CIKARTMA (Connected Components)|")
print(f"  |     Min alan: {MIN_NODULE_AREA} piksel                   |")
print("  |     Bbox + merkez + cap hesapla           |")
print("  +------------------------------------------+")
print("        |")
print("        v")
print("  +------------------------------------------+")
print("  |  4. KARAKTERIZASYON (ResNet-50 + CBAM)   |")
print(f"  |     Giris: 3ch x {CHAR_PATCH_SIZE} x {CHAR_PATCH_SIZE} (nodul patch)  |")
print("  |     Cikis 1: Suspicious (binary)          |")
print("  |     Cikis 2: Risk sinifi (4 sinif)        |")
print("  +------------------------------------------+")
print("        |")
print("        v")
print("  +------------------------------------------+")
print("  |  5. KLINIK KARAR DESTEGI                 |")
print("  |     Lung-RADS siniflandirma               |")
print("  |     Risk tabanlasi takip onerisi           |")
print("  +------------------------------------------+")
print("        |")
print("        v")
print("  [Yapilandirilmis Rapor (JSON/CSV)]")
print()

In [ ]:
# Hasta bazinda ozet tablosu (renk kodlamali)
if len(detected_df) > 0:
    fig, ax = plt.subplots(figsize=(14, max(4, n_test_patients * 0.4)))
    ax.axis('off')
    
    # Tablo verisi
    table_data = []
    cell_colors = []
    
    header = ['Hasta', 'Nodul', 'Tespit', 'Dice', 'Suspicious', 'Risk', 'Lung-RADS']
    
    for pid in results_df['patient_id'].unique():
        pdf = results_df[results_df['patient_id'] == pid]
        n_nodules = len(pdf)
        n_det = int(pdf['detected'].sum())
        mean_dice = pdf['dice'].mean()
        
        det_pdf = pdf[pdf['detected'] == True]
        if len(det_pdf) > 0:
            mean_susp = det_pdf['suspicious_prob'].mean()
            # En yuksek risk sinifini goster
            max_risk = int(det_pdf['risk_class'].max())
            risk_name = RISK_NAMES[max_risk]
            lr = RISK_TO_LUNGRADS.get(max_risk, '?')
        else:
            mean_susp = 0
            risk_name = '-'
            lr = '-'
            max_risk = -1
        
        short_pid = pid.replace('LIDC-IDRI-', '')
        row_data = [
            short_pid,
            f'{n_nodules}',
            f'{n_det}/{n_nodules}',
            f'{mean_dice:.3f}',
            f'{mean_susp:.2f}',
            risk_name,
            f'LR-{lr}',
        ]
        table_data.append(row_data)
        
        # Renk kodlama
        if max_risk == 3:
            row_color = ['#ffcccc'] * len(header)  # Kirmizi
        elif max_risk == 2:
            row_color = ['#ffe0b2'] * len(header)  # Turuncu
        elif max_risk == 1:
            row_color = ['#fff9c4'] * len(header)  # Sari
        elif max_risk == 0:
            row_color = ['#c8e6c9'] * len(header)  # Yesil
        else:
            row_color = ['#e0e0e0'] * len(header)  # Gri
        cell_colors.append(row_color)
    
    # Header rengi
    header_color = [['#1565c0'] * len(header)]
    
    table = ax.table(
        cellText=table_data,
        colLabels=header,
        cellColours=cell_colors,
        colColours=['#1565c0'] * len(header),
        cellLoc='center',
        loc='center',
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.5)
    
    # Header metin rengi beyaz
    for j in range(len(header)):
        table[0, j].get_text().set_color('white')
        table[0, j].get_text().set_fontweight('bold')
    
    ax.set_title('Hasta Bazinda Pipeline Sonuclari', fontsize=13, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ct_pipeline_patient_table.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Kaydedildi: ct_pipeline_patient_table.png")
else:
    print("Gorsellestirme icin yeterli veri yok.")

In [ ]:
# Zamanlama analizi
timing_df = pd.DataFrame(all_timings)

print("=" * 60)
print("Zamanlama Analizi")
print("=" * 60)
print(f"\nHasta bazinda ortalama sureler:")
print(f"  Segmentasyon:    {timing_df['segment'].mean():.3f}s")
print(f"  CC Cikartma:     {timing_df['extract'].mean():.3f}s")
print(f"  Karakterizasyon: {timing_df['characterize'].mean():.3f}s")
print(f"  Toplam:          {timing_df['total'].mean():.3f}s")

# Zamanlama grafigi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Asama bazinda zamanlama
stages = ['Segmentasyon', 'CC Cikartma', 'Karakterizasyon']
stage_means = [
    timing_df['segment'].mean(),
    timing_df['extract'].mean(),
    timing_df['characterize'].mean(),
]
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = axes[0].bar(stages, stage_means, color=colors, edgecolor='white')
for bar_obj, val in zip(bars, stage_means):
    axes[0].text(
        bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 0.001,
        f'{val:.3f}s', ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
axes[0].set_title('Asama Bazinda Ortalama Sure', fontweight='bold')
axes[0].set_ylabel('Sure (saniye)')

# Sag: Lung-RADS dagilimi
if len(detected_df) > 0:
    lr_order = ['2', '3', '4A', '4B']
    lr_counts = detected_df['lung_rads'].value_counts()
    lr_vals = [lr_counts.get(lr, 0) for lr in lr_order]
    lr_colors = RISK_COLORS
    
    bars2 = axes[1].bar(
        [f'LR-{lr}' for lr in lr_order], lr_vals,
        color=lr_colors, edgecolor='black', linewidth=0.5
    )
    for bar_obj, val in zip(bars2, lr_vals):
        if val > 0:
            axes[1].text(
                bar_obj.get_x() + bar_obj.get_width() / 2, bar_obj.get_height() + 0.3,
                str(val), ha='center', va='bottom', fontsize=11, fontweight='bold'
            )
    axes[1].set_title('Lung-RADS Dagilimi', fontweight='bold')
    axes[1].set_ylabel('Nodul Sayisi')
else:
    axes[1].text(0.5, 0.5, 'Veri yok', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ct_pipeline_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Kaydedildi: ct_pipeline_risk_distribution.png")

---
## 8. Klinik Karar Destegi

In [ ]:
# Risk tabakalamasina gore hasta gruplama ve takip onerileri

FOLLOWUP_RECOMMENDATIONS = {
    '2':  'Rutin tarama -- 12 aylik BT kontrolu yeterli.',
    '3':  '6 aylik takip BT onerilir. Boyut degisimi yakından izlenmeli.',
    '4A': '3 aylik takip BT veya PET/CT onerilir. Klinik korelasyon gerekli.',
    '4B': 'Doku orneklemesi (biyopsi) onerilir. Multidisipliner konsey degerlendirmesi.',
}

print("=" * 70)
print("Klinik Karar Destek Raporu")
print("=" * 70)

if len(detected_df) > 0:
    # Lung-RADS bazinda gruplama
    for lr_cat in ['2', '3', '4A', '4B']:
        lr_group = detected_df[detected_df['lung_rads'] == lr_cat]
        if len(lr_group) == 0:
            continue
        
        n_patients = lr_group['patient_id'].nunique()
        n_nodules = len(lr_group)
        
        print(f"\n--- Lung-RADS {lr_cat}: {LUNGRADS_DESCRIPTIONS[lr_cat]} ---")
        print(f"  Hasta sayisi: {n_patients}")
        print(f"  Nodul sayisi: {n_nodules}")
        print(f"  Ortalama susp. olasiligi: {lr_group['suspicious_prob'].mean():.3f}")
        print(f"  Ortalama nodul capi: {lr_group['pred_diameter_px'].mean():.1f} piksel")
        print(f"  Oneri: {FOLLOWUP_RECOMMENDATIONS[lr_cat]}")
        
        # Hastalari listele
        for pid in lr_group['patient_id'].unique():
            p_nodules = lr_group[lr_group['patient_id'] == pid]
            max_susp = p_nodules['suspicious_prob'].max()
            short_pid = pid.replace('LIDC-IDRI-', '')
            print(f"    {short_pid}: {len(p_nodules)} nodul, max suspicious={max_susp:.2f}")
else:
    print("  Tespit edilen nodul yok.")

print(f"\n{'=' * 70}")
print(f"UYARI: Bu sonuclar sadece arastirma amaclıdır.")
print(f"Klinik karar icin uzman radyolog degerlendirmesi sarttir.")
print(f"{'=' * 70}")

In [ ]:
# Her hasta icin yapilandirilmis JSON raporu olustur

patient_reports = {}

for pid in results_df['patient_id'].unique():
    pdf = results_df[results_df['patient_id'] == pid]
    
    nodule_reports = []
    for _, row in pdf.iterrows():
        nodule_report = {
            'nodule_id': row['nodule'],
            'n_slices': int(row['n_slices']),
            'n_annotators': int(row['n_annotators']),
            'detected': bool(row['detected']),
            'segmentation': {
                'dice_score': round(float(row['dice']), 4),
                'gt_area_px': int(row['gt_area']),
                'pred_area_px': int(row['pred_area']),
                'gt_diameter_px': round(float(row['gt_diameter_px']), 1),
                'pred_diameter_px': round(float(row['pred_diameter_px']), 1),
            },
        }
        
        if row['detected']:
            rp = row.get('risk_probs', [])
            if not isinstance(rp, list):
                rp = []
            nodule_report['characterization'] = {
                'suspicious_probability': round(float(row['suspicious_prob']), 4),
                'suspicious': bool(row['suspicious']),
                'risk_class': int(row['risk_class']),
                'risk_name': row['risk_name'],
                'lung_rads': row['lung_rads'],
                'lung_rads_description': LUNGRADS_DESCRIPTIONS.get(row['lung_rads'], ''),
                'followup_recommendation': FOLLOWUP_RECOMMENDATIONS.get(row['lung_rads'], ''),
                'risk_probabilities': {
                    'low': round(rp[0], 4) if len(rp) > 0 else 0.0,
                    'mid_low': round(rp[1], 4) if len(rp) > 1 else 0.0,
                    'mid_high': round(rp[2], 4) if len(rp) > 2 else 0.0,
                    'high': round(rp[3], 4) if len(rp) > 3 else 0.0,
                },
            }
        
        nodule_reports.append(nodule_report)
    
    # Hasta icin en yuksek risk seviyesi
    detected_nodules = [n for n in nodule_reports if n['detected']]
    if detected_nodules:
        max_risk = max(
            n['characterization']['risk_class'] for n in detected_nodules
        )
        overall_lr = RISK_TO_LUNGRADS.get(max_risk, '?')
    else:
        max_risk = -1
        overall_lr = '1'  # Nodul tespit edilemedi
    
    patient_reports[pid] = {
        'patient_id': pid,
        'n_nodules_total': len(pdf),
        'n_nodules_detected': int(pdf['detected'].sum()),
        'overall_lung_rads': overall_lr,
        'overall_risk': RISK_NAMES[max_risk] if max_risk >= 0 else 'Nodul tespit edilemedi',
        'followup': FOLLOWUP_RECOMMENDATIONS.get(overall_lr, 'Rutin tarama.'),
        'nodules': nodule_reports,
    }

# JSON olarak kaydet
with open(OUTPUT_DIR / 'ct_pipeline_patient_reports.json', 'w', encoding='utf-8') as f:
    json.dump(patient_reports, f, indent=2, ensure_ascii=False)

print(f"Hasta raporlari kaydedildi: ct_pipeline_patient_reports.json")
print(f"  Toplam hasta: {len(patient_reports)}")

# Ornek rapor goster
sample_pid = list(patient_reports.keys())[0]
print(f"\nOrnek rapor ({sample_pid}):")
print(json.dumps(patient_reports[sample_pid], indent=2, ensure_ascii=False))

---
## 9. Cikti ve Ozet

In [ ]:
# === CSV kaydet ===
# risk_probs kolonu liste icerir, string olarak kaydet
save_df = results_df.copy()
save_df['risk_probs'] = save_df['risk_probs'].apply(str)
save_df.to_csv(OUTPUT_DIR / 'ct_pipeline_results.csv', index=False)
print(f"Pipeline sonuclari kaydedildi: ct_pipeline_results.csv")
print(f"  Satir: {len(save_df)}, Sutun: {save_df.shape[1]}")

# === Pipeline konfigurasyonu kaydet ===
pipeline_config = {
    'pipeline_version': '1.0',
    'segmentation': {
        'model': 'smp.Unet (ResNet-34 encoder)',
        'input_size': SEG_IMG_SIZE,
        'threshold': SEG_THRESHOLD,
        'min_nodule_area': MIN_NODULE_AREA,
        'weights_file': 'ct_seg_best_model.pth',
        'n_parameters': n_seg_params,
    },
    'characterization': {
        'model': 'ResNet-50 + CBAM',
        'input_size': CHAR_PATCH_SIZE,
        'suspicious_threshold': SUSP_THRESHOLD,
        'n_risk_classes': 4,
        'weights_file': 'ct_char_best_model.pth',
        'n_parameters': n_char_params,
    },
    'lung_rads_mapping': RISK_TO_LUNGRADS,
    'followup_recommendations': FOLLOWUP_RECOMMENDATIONS,
    'risk_names': RISK_NAMES,
}

with open(OUTPUT_DIR / 'ct_pipeline_config.json', 'w', encoding='utf-8') as f:
    json.dump(pipeline_config, f, indent=2, ensure_ascii=False)
print(f"Pipeline konfigurasyonu kaydedildi: ct_pipeline_config.json")

# === Metrikler CSV ===
metrics_data = {
    'metric': [
        'detection_rate',
        'mean_dice_detected',
        'mean_dice_all',
        'n_patients',
        'n_nodules_total',
        'n_nodules_detected',
    ],
    'value': [
        round(detection_rate, 4),
        round(float(detected_df['dice'].mean()) if len(detected_df) > 0 else 0.0, 4),
        round(float(results_df['dice'].mean()), 4),
        n_test_patients,
        n_total,
        n_detected,
    ],
}

# Risk ve AUC metrikleri ekle (eger hesaplanabildiyse)
if not pd.isna(risk_acc) if isinstance(risk_acc, float) else True:
    metrics_data['metric'].append('risk_accuracy')
    metrics_data['value'].append(round(float(risk_acc), 4) if not pd.isna(risk_acc) else 0.0)

if not pd.isna(auc) if isinstance(auc, float) else True:
    metrics_data['metric'].append('suspicious_auc')
    metrics_data['value'].append(round(float(auc), 4) if not pd.isna(auc) else 0.0)

metrics_data['metric'].append('avg_time_per_patient_s')
metrics_data['value'].append(round(total_time / n_test_patients, 3))

metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv(OUTPUT_DIR / 'ct_pipeline_metrics.csv', index=False)
print(f"Pipeline metrikleri kaydedildi: ct_pipeline_metrics.csv")

In [ ]:
# Cikti dosyalarini listele
print("=" * 60)
print("Kaydedilen Dosyalar")
print("=" * 60)

output_files = sorted(OUTPUT_DIR.glob('ct_pipeline_*'))
total_size = 0
for f in output_files:
    size_bytes = f.stat().st_size
    total_size += size_bytes
    if size_bytes >= 1024 * 1024:
        size_str = f"{size_bytes / (1024 * 1024):.1f} MB"
    else:
        size_str = f"{size_bytes / 1024:.1f} KB"
    print(f"  {f.name} ({size_str})")

print(f"\nToplam: {len(output_files)} dosya, {total_size / 1024:.1f} KB")

In [ ]:
# ==========================================
# FINAL OZET
# ==========================================

print("\n" + "=" * 70)
print("AlpCAN CT Pipeline -- Uctan Uca Nodul Analizi FINAL OZET")
print("=" * 70)

print(f"\n--- Veri Seti ---")
print(f"  LIDC-IDRI (zhangweiled/lidcidri)")
print(f"  Test hasta sayisi: {n_test_patients}")
print(f"  Toplam nodul: {n_total}")

print(f"\n--- Modeller ---")
print(f"  Segmentasyon: U-Net (ResNet-34) -- {n_seg_params/1e6:.1f}M parametre")
print(f"  Karakterizasyon: ResNet-50 + CBAM -- {n_char_params/1e6:.1f}M parametre")
print(f"  Toplam: {(n_seg_params + n_char_params)/1e6:.1f}M parametre")

print(f"\n--- Pipeline Performansi ---")
print(f"  Tespit orani: {detection_rate:.1%} ({n_detected}/{n_total})")
if len(detected_df) > 0:
    print(f"  Ort. Dice (tespit edilen): {detected_df['dice'].mean():.4f}")
print(f"  Ort. Dice (tum noduller): {results_df['dice'].mean():.4f}")

if not pd.isna(risk_acc) if isinstance(risk_acc, float) else False:
    print(f"  Risk Accuracy: {risk_acc:.4f}")
if not pd.isna(auc) if isinstance(auc, float) else False:
    print(f"  Suspicious AUC: {auc:.4f}")

print(f"\n--- Zamanlama ---")
print(f"  Toplam sure: {total_time:.1f}s ({total_time/60:.1f} dk)")
print(f"  Hasta basina: {total_time/n_test_patients:.2f}s")
print(f"  Segmentasyon ort.: {timing_df['segment'].mean():.3f}s")
print(f"  Karakterizasyon ort.: {timing_df['characterize'].mean():.3f}s")

print(f"\n--- Lung-RADS Dagilimi ---")
if len(detected_df) > 0:
    for lr_cat in ['2', '3', '4A', '4B']:
        count = int((detected_df['lung_rads'] == lr_cat).sum())
        if count > 0:
            print(f"  Lung-RADS {lr_cat}: {count} nodul -- {LUNGRADS_DESCRIPTIONS[lr_cat]}")

print(f"\n--- Klinik Entegrasyon ---")
print(f"  Pipeline JSON raporlari: ct_pipeline_patient_reports.json")
print(f"  Her hasta icin: Lung-RADS sinifi + takip onerisi")
print(f"  DICOM entegrasyonu: Orthanc uzerinden AlpCAN backend'e aktarilabilir")

print(f"\n--- Cikti Dosyalari ---")
print(f"  ct_pipeline_results.csv              -- Tum nodul sonuclari")
print(f"  ct_pipeline_config.json              -- Pipeline konfigurasyonu")
print(f"  ct_pipeline_patient_reports.json      -- Hasta raporlari")
print(f"  ct_pipeline_visualization.png         -- 8 hasta gorsellestirme")
print(f"  ct_pipeline_metrics.csv               -- Pipeline metrikleri")
print(f"  ct_pipeline_confusion_matrix.png      -- Risk confusion matrix")
print(f"  ct_pipeline_risk_distribution.png     -- Lung-RADS dagilimi")

print(f"\n--- Sonraki Adimlar ---")
print(f"  1. Model agirliklarini AlpCAN backend'e yukle")
print(f"  2. ct_pipeline_inference.py modulu olustur")
print(f"  3. Orthanc DICOM entegrasyonu")
print(f"  4. CXR + CT pipeline birlesik karar destek sistemi")
print(f"  5. 3D volumetrik analiz (tam BT serisi)")
print("=" * 70)

---

## Sonuc ve Degerlendirme

Bu notebook'ta AlpCAN CT Pipeline'inin uctan uca calismasi gosterilmistir:

1. **Segmentasyon Asamasi:** U-Net (ResNet-34 encoder) modeli ile BT goruntusundeki noduller otomatik olarak segmente edilmistir.

2. **Nodul Cikartma:** Connected component analizi ile segmentasyon maskesinden bireysel nodul adaylari cikarilmistir.

3. **Karakterizasyon Asamasi:** ResNet-50 + CBAM modeli ile her nodul adayi siniflandirilmis, suspicious (supheli/benign) ve risk sinifi (4 seviye) tahminleri uretilmistir.

4. **Klinik Karar Destegi:** Lung-RADS siniflandirmasi ile her hasta icin takip onerileri olusturulmustur.

### Temel Bulgular:
- Pipeline uctan uca calisabilir durumda
- Segmentasyon ve karakterizasyon modelleri birlikte kullanildiginda anlamli sonuclar uretmektedir
- Lung-RADS esleme ile klinik karar destegi saglanmaktadir

### Sinirliliklar:
- 2D dilim bazli analiz (3D volumetrik degil)
- LIDC-IDRI kesilmis dilim formati kullanilmistir
- Gercek DICOM verisi ile entegrasyon henuz yapilmamistir
- Boyut bazli risk siniflandirmasi (gercek malignite etiketi degil)

### AlpCAN Entegrasyonu:
Bu pipeline AlpCAN backend'e entegre edilecek ve Orthanc DICOM sunucusu uzerinden gelen BT serilerini otomatik analiz edecektir.